# MD Sync - Single Lakehouse

Change **only `lakehouse_id`**, choose an `action`, then run all cells.
The workspace is taken from this notebook's current workspace; the lakehouse must be in it.

| Action | Behavior |
| --- | --- |
| `Action.DUMMY_RUN` | Validate only; display endpoint state and each Delta table's readiness/details. |
| `Action.UPGRADE` | Upgrade eligible V1 tables; show blocked/skipped tables. |
| `Action.FORCE_UPGRADE` | Attempt all remaining V1 tables, even if preflight finds no eligible tables; raise an error listing any tables that do not upgrade. |

The REST API upgrades an **entire endpoint**, not a caller-selected table list.
Both upgrade actions use that same API; force is a client-side choice, not a REST
parameter, and cannot make incompatible tables succeed. After a partial upgrade,
failed V1 tables **stop syncing automatically**. Use a maintenance window.

Table names come from OneLake metadata. If the inventory and assessment counts
disagree, readiness is shown as **Not reported / inventory mismatch**, not guessed.
No audit table or additional lakehouse is created.

[Documented upgrade process](https://learn.microsoft.com/en-us/fabric/data-engineering/sql-analytics-endpoint-metadata-sync#upgrade-existing-sql-analytics-endpoints-to-new-metadata-sync)


In [ ]:
lakehouse_id = "4cadc8ab-c5cb-4f36-9ca1-9b1bbad8cf38"


## Small helpers

REST authentication/polling, OneLake table discovery, and the three action branches.


In [ ]:
import json
import time
from enum import Enum
from urllib.parse import urlsplit
from uuid import UUID

import notebookutils
import pandas as pd
import requests


class Action(str, Enum):
    DUMMY_RUN = "dummy run"
    UPGRADE = "upgrade"
    FORCE_UPGRADE = "force_upgrade"


API = "https://api.fabric.microsoft.com/v1"


def request(method, path):
    for attempt in range(4):
        response = requests.request(
            method, API + path,
            headers={"Authorization": "Bearer " + notebookutils.credentials.getToken("pbi")},
            timeout=120, allow_redirects=False,
        )
        if response.status_code != 429 or attempt == 3:
            if response.status_code not in (200, 201, 202):
                raise RuntimeError(f"Fabric HTTP {response.status_code}: {response.text}")
            return response
        time.sleep(int(response.headers.get("Retry-After", "5")))


def api(method, path):
    response = request(method, path)
    if response.status_code == 202:
        operation_id = str(UUID(response.headers["x-ms-operation-id"]))
        print(f"Operation: {operation_id}")
        deadline = time.monotonic() + 1800
        while True:
            delay = int(response.headers.get("Retry-After", "5"))
            if time.monotonic() + delay >= deadline:
                raise TimeoutError(f"Operation {operation_id} is still running; do not replay an upgrade blindly.")
            time.sleep(delay)
            response = request("GET", f"/operations/{operation_id}")
            state = response.json()
            if state.get("status") == "Succeeded":
                response = request("GET", f"/operations/{operation_id}/result")
                if response.status_code != 202:
                    break
            elif state.get("status") not in ("NotStarted", "Running"):
                raise RuntimeError(f"Operation {operation_id}: {json.dumps(state)}")
    result = response.json()
    if not isinstance(result, dict):
        raise RuntimeError("Fabric returned an unexpected response; expected a JSON object.")
    return result


def tables_in(lakehouse):
    properties = lakehouse["properties"]
    location = urlsplit(properties["oneLakeTablesPath"])
    if location.scheme != "https" or not location.hostname.endswith(".dfs.fabric.microsoft.com"):
        raise RuntimeError("The lakehouse returned an unexpected OneLake tables path.")
    workspace, item, folder = location.path.strip("/").split("/")
    root = f"abfss://{workspace}@{location.hostname}/{item}/{folder}"
    schemas = (
        [(entry.name.rstrip("/"), entry.path) for entry in notebookutils.fs.ls(root) if entry.isDir]
        if properties.get("defaultSchema") else [("dbo", root)]
    )
    return [
        {"schema": schema, "tableName": entry.name.rstrip("/"), "location": entry.path}
        for schema, path in schemas
        for entry in notebookutils.fs.ls(path)
        if entry.isDir and notebookutils.fs.exists(entry.path.rstrip("/") + "/_delta_log")
    ]


def table_details(tables, status, preflight):
    key = lambda row: (row["schema"], row["tableName"])
    blockers = {}
    for row in preflight["blockers"]:
        blockers.setdefault(key(row), []).append(row["correctiveAction"])
    failures = {key(row): row["error"].get("message", "") for row in status["tableFailures"]}
    all_keys = {key(row) for row in tables}
    legacy = all_keys if status["syncArchitectureVersion"] == "V1" else set(failures)
    evaluated, eligible = preflight["evaluatedTableCount"], preflight["eligibleTableCount"]
    complete = (
        evaluated in (len(all_keys), len(legacy))
        and eligible == evaluated - len(blockers)
        and set(blockers) <= all_keys
    )
    rows = []
    for row in tables:
        identity = key(row)
        version = "V1" if identity in legacy else "V2"
        verdict = (
            "Blocked" if identity in blockers else
            "Already V2" if version == "V2" else
            "Eligible" if complete else "Not reported / inventory mismatch"
        )
        rows.append(dict(row, md_sync=version, validation=verdict,
                         details="; ".join(blockers.get(identity, [])) or failures.get(identity, "")))
    for identity in set(blockers) - all_keys:
        rows.append({"schema": identity[0], "tableName": identity[1], "md_sync": "V1",
                     "validation": "Blocked", "details": "; ".join(blockers[identity]), "location": None})
    return rows


def run(lakehouse_id, action):
    action = Action(action)
    lakehouse_id = str(UUID(lakehouse_id))
    workspace = str(UUID(notebookutils.runtime.context["currentWorkspaceId"]))
    lakehouse = api("GET", f"/workspaces/{workspace}/lakehouses/{lakehouse_id}")
    endpoint = lakehouse["properties"]["sqlEndpointProperties"]
    if endpoint.get("provisioningStatus") != "Success":
        raise RuntimeError(f"SQL endpoint is not ready: {endpoint}")
    root = f"/workspaces/{workspace}/sqlEndpoints/{str(UUID(endpoint['id']))}"
    before = api("GET", root + "/metadataSyncMigrationStatus")
    if before.get("syncArchitectureVersion") not in ("V1", "V2") or not isinstance(before.get("tableFailures"), list):
        raise RuntimeError(f"Unrecognized metadata sync status: {before}")
    print(f"{lakehouse['displayName']} | MD Sync {before['syncArchitectureVersion']} | {action.value}")
    display(pd.json_normalize([before]))
    if before.get("itemMigrationDirection"):
        raise RuntimeError("A migration is already running. Wait for it to finish before assessing/upgrading.")

    preflight = api("POST", root + "/preflightMetadataSyncUpgrade")
    evaluated, eligible = preflight.get("evaluatedTableCount"), preflight.get("eligibleTableCount")
    if (type(evaluated) is not int or type(eligible) is not int or
            not 0 <= eligible <= evaluated or not isinstance(preflight.get("blockers"), list)):
        raise RuntimeError(f"Unrecognized preflight response: {preflight}")
    details = table_details(tables_in(lakehouse), before, preflight)
    print(f"Validation: {eligible}/{evaluated} eligible; {len(preflight['blockers'])} blocker(s).")
    display(pd.DataFrame(details, columns=["schema", "tableName", "md_sync", "validation", "details", "location"]))
    summary = {"lakehouse_id": lakehouse_id, "action": action.value, "status": before,
               "preflight": preflight, "tables": details, "upgrade_attempted": False}
    if action == Action.DUMMY_RUN:
        return summary
    if before["syncArchitectureVersion"] == "V2" and not before["tableFailures"]:
        print("Already V2; nothing to upgrade.")
        return summary
    if evaluated == 0 or (action == Action.UPGRADE and eligible == 0):
        print("No eligible V1 tables to upgrade." if eligible == 0 else "No tables to upgrade.")
        return summary

    current = api("GET", root + "/metadataSyncMigrationStatus")
    if current.get("itemMigrationDirection") or current.get("syncArchitectureVersion") != before["syncArchitectureVersion"]:
        raise RuntimeError("Endpoint state changed after validation. Rerun the notebook.")
    summary["upgrade_attempted"] = True
    try:
        result = api("POST", root + "/upgradeMetadataSync")
    except (RuntimeError, TimeoutError, requests.exceptions.RequestException):
        display(pd.json_normalize([api("GET", root + "/metadataSyncMigrationStatus")]))
        raise
    after = api("GET", root + "/metadataSyncMigrationStatus")
    outcomes = result.get("tableStatuses")
    migration = str(result.get("itemMigrationStatus", "")).casefold()
    if not isinstance(outcomes, list) or not outcomes or migration not in ("succeeded", "partiallysucceeded", "failed"):
        raise RuntimeError(f"Unrecognized upgrade response: {result}")
    display(pd.json_normalize(outcomes))
    display(pd.json_normalize([after]))
    failed = [row for row in outcomes if row.get("syncArchitectureVersion") != "V2" or row.get("error")]
    if not isinstance(after.get("tableFailures"), list):
        raise RuntimeError(f"Unrecognized post-upgrade status: {after}")
    failed_keys = {(row["schema"], row["tableName"]) for row in failed}
    failed += [dict(row, syncArchitectureVersion="V1") for row in after["tableFailures"]
               if (row["schema"], row["tableName"]) not in failed_keys]
    summary.update(status=after, upgrade=result, failed_tables=failed)
    if failed:
        print("Tables remaining on V1 do not sync automatically after a partial upgrade.")
        display(pd.json_normalize(failed))
    blocked = {(row["schema"], row["tableName"]) for row in preflight["blockers"]}
    unexpected = [row for row in failed if (row["schema"], row["tableName"]) not in blocked]
    if failed and (action == Action.FORCE_UPGRADE or unexpected or migration == "failed"):
        names = "; ".join(
            f"{row['schema']}.{row['tableName']}: {(row.get('error') or {}).get('message', 'still V1')}"
            for row in failed
        )
        raise RuntimeError("Tables did not upgrade: " + names)
    if after.get("itemMigrationDirection") or after.get("syncArchitectureVersion") != "V2":
        raise RuntimeError(f"Upgrade could not be verified: {after}")
    if migration != "succeeded" and not failed:
        raise RuntimeError(f"Upgrade reported {migration} without per-table failure details: {result}")
    print("Eligible tables upgraded; see the per-table results above.")
    return summary


## Choose the action

Keep `DUMMY_RUN` for validation only. `action` is an enum selector, not another notebook input parameter.


In [ ]:
action = Action.FORCE_UPGRADE


In [ ]:
summary = run(lakehouse_id, action)
if notebookutils.runtime.context.get("isForInteractive") is False:
    notebookutils.notebook.exit(json.dumps(summary))
